# Cuaderno 11 — Circularidad

**Qué pregunta responde.** El segundo reparo de la tutora es el más serio de los cinco y es el
único que sigue abierto: si las variables y el índice se eligieron de modo que produjeran el
resultado, entonces el resultado no dice nada sobre las compras públicas, solo sobre cómo lo
construimos. Es una objeción legítima y la prueba de permutación del cuaderno 8 **no la
responde**: esa prueba muestra que el procedimiento no fabrica señal cuando las etiquetas son
falsas, pero no dice nada sobre cómo se eligieron las variables.

Este cuaderno tiene tres piezas, cada una contra una forma distinta de circularidad:

| Pieza | Contra qué | Qué produce |
| --- | --- | --- |
| 1 · Declaración de especificación | «cambiaron las variables hasta que salió» | Firma verificable de cada conjunto, con sello de tiempo |
| 2 · Índice solo con literatura | «el índice está hecho a la medida de la etiqueta» | Un índice con regla escrita de antemano, contra un nulo de índices al azar |
| 3 · Especificación por mitades | «la especificación está pegada a esos 48 casos» | El valor en una mitad que no se usó para elegir |

**Una advertencia que conviene leer antes de correr nada.** Ninguna de las tres convierte este
trabajo en una preinscripción. La especificación no se declaró antes de ver resultados, porque
no se trabajó así: se revisó dos veces después de mirar. Lo honesto no es inventar una
preinscripción retroactiva, es declarar las dos revisiones con su fecha y su dirección. Y la
dirección importa: en los dos casos quitar la variable **mejoró o no cambió** el resultado, que
es lo contrario de lo que produce elegir para que salga.


In [ ]:
# ============ Entorno · protocolo identico al del paso 12 del cuaderno 8 ============
import os, sys, json, time, io, contextlib, hashlib, datetime
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
os.chdir(r'C:\\SERCOP_FINAL')
R = '03_resultados'; OUT = f'{R}/c8_'; OUT9 = f'{R}/c9_'
SEMILLA = 20261001

# Los cuatro numeros del protocolo. Son los MISMOS del paso 12: si el estadistico no se
# calcula igual aqui que alla, ninguna comparacion de este cuaderno vale nada.
K_P, RATIO_U, N_MUESTRA, SEM_MUESTRA = 15, 5, 600, 777

def pu_bagging(Xp, Xu, Xs, k=K_P, ratio=RATIO_U, seed=0):
    r = np.random.default_rng(seed); acum = np.zeros(len(Xs)); n = len(Xp); hechos = 0
    with contextlib.redirect_stderr(io.StringIO()):
        for i in range(k):
            idx = r.choice(len(Xu), size=min(len(Xu), n*ratio), replace=False)
            Xtr = np.vstack([Xp, Xu[idx]]); ytr = np.r_[np.ones(n), np.zeros(len(idx))]
            util = np.array([np.unique(Xtr[~np.isnan(Xtr[:,j]), j]).size >= 2 for j in range(Xtr.shape[1])])
            if util.sum() == 0: continue
            m = HistGradientBoostingClassifier(max_depth=3, max_iter=60, learning_rate=0.1,
                                               min_samples_leaf=5, l2_regularization=1.0,
                                               random_state=i).fit(Xtr[:,util], ytr)
            acum += m.predict_proba(Xs[:,util])[:,1]; hechos += 1
    assert hechos > 0
    return acum / hechos

def muestra_comparacion(y, anios, semilla=SEM_MUESTRA, n_por_anio=N_MUESTRA):
    r = np.random.default_rng(semilla); partes = []
    for a in np.unique(anios):
        m = np.where((y == 0) & (anios == a))[0]
        if len(m): partes.append(r.choice(m, size=min(n_por_anio, len(m)), replace=False))
    return np.concatenate(partes)

def estadistico(X, y, gi, anios, n_folds, k, semilla, Xu=None):
    """Percentil mediano dentro del anio de los positivos. Cada positivo lo puntua un modelo
    que NO vio su conglomerado."""
    r  = np.random.default_rng(semilla)
    gp = np.unique(gi[y == 1])
    if len(gp) < 2: return np.nan
    idx_s = np.concatenate([np.where(y == 1)[0], muestra_comparacion(y, anios)])
    Xs = X[idx_s]
    if Xu is None: Xu = X[y == 0]
    score = np.full(len(idx_s), np.nan); acum = np.zeros(len(idx_s)); nf = 0
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(r.permutation(gp), min(n_folds, len(gp))) if len(f)])
    for j, fuera in enumerate(folds):
        dentro = np.isin(gi, fuera)
        s = pu_bagging(X[(y == 1) & ~dentro], Xu, Xs, k=k, seed=semilla + j)
        donde = dentro[idx_s] & (y[idx_s] == 1)
        score[donde] = s[donde]
        acum += s; nf += 1
    falta = np.isnan(score); score[falta] = (acum / nf)[falta]
    ev  = pd.DataFrame({'y': y[idx_s], 'anio': anios[idx_s], 'm': score})
    pct = ev.groupby('anio')['m'].rank(pct=True).values
    return float(np.nanmedian(pct[ev.y.values == 1]))

def pct_anio(d, col):
    """Percentil dentro del anio de una columna. Es como el paso 12 evalua los indices."""
    return d.groupby('anio_proceso')[col].rank(pct=True).values

def promedio_seguro(M):
    """Promedio por fila ignorando nulos, sin el aviso de numpy cuando la fila es toda nula."""
    n = np.sum(~np.isnan(M), axis=1); s = np.nansum(M, axis=1)
    out = np.full(M.shape[0], np.nan); ok = n > 0
    out[ok] = s[ok] / n[ok]
    return out

def pct_anio_arr(x, anios):
    return pd.DataFrame({'x': x, 'a': anios}).groupby('a')['x'].rank(pct=True).values

def stat_indice(pz, y):
    return float(np.nanmedian(np.asarray(pz)[np.asarray(y) == 1]))

P = pd.read_parquet(f'{OUT}panel.parquet')
with open(f'{OUT}vars.json', encoding='utf-8') as f:
    VARS = json.load(f)
print(f'panel: {P.shape[0]:,} filas x {P.shape[1]} columnas')
print(f'positivos A: {int(P.y_fantasma.sum())} en '
      f'{P.loc[P.y_fantasma == 1, "proveedor_ruc"].nunique()} conglomerados')
print('conjuntos de variables:', {k: len(v) for k, v in VARS.items()})

## Pieza 1 · Declaración de especificación

Las dos revisiones que hubo, con lo que se sabe de cada una:

| Cuándo | Qué cambió | Por qué | Efecto en el resultado |
| --- | --- | --- | --- |
| Septiembre | Se excluyeron las variables derivadas de la marca del SRI | Son la etiqueta misma: `proveedor_fantasma_estado` predice `y_fantasma` por definición | Bajó, como debía |
| Septiembre | Se rehízo el emparejamiento del objetivo B desde el directorio completo de la Superintendencia | La versión anterior partía de pares que ya competían: circular por construcción | 35 pares → 446 procesos |
| 6 de octubre | Se excluyó `monto_tender` | Su patrón de nulos **es** el tipo de procedimiento: canal lateral hacia el procedimiento | **Subió**: 0,760 → 0,797 en el universo restringido |

La tercera es la que más sirve como argumento. Si hubiéramos estado eligiendo variables para que
el resultado saliera, no habríamos quitado una que lo bajaba de 0,797 a 0,760 —la habríamos
dejado—. La quitamos porque su origen la descalifica, y el resultado mejoró. Esa es la regla de
exclusión del trabajo, escrita desde el principio: **la exclusión se decide por origen, no por
desempeño.**

La prueba de que la regla no es retórica está en el cuaderno 10, entrada F03:
`ratio_monto_capital` tiene AUC 0,820 y **se conserva**, porque su origen es legítimo; un
indicador de disolución tiene AUC 0,967 y **se excluye**, porque viene de la misma fuente que la
etiqueta. Si la regla fuera «quedarse con lo que funciona», esas dos decisiones serían las
contrarias.

La celda de abajo firma los ocho conjuntos de variables con SHA-256 y guarda el sello de tiempo,
las huellas de los archivos de datos y los cuatro números del protocolo. De aquí en adelante
cualquier cambio en las variables cambia la firma y queda a la vista.


In [ ]:
# --- Pieza 1: congelar y firmar la especificacion ---
def firma(lista):
    return hashlib.sha256('\n'.join(sorted(lista)).encode('utf-8')).hexdigest()[:16]

sello = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
esp = {
    'sello': sello,
    'semilla_global': SEMILLA,
    'protocolo': {'k_bolsas': K_P, 'ratio_no_etiquetados': RATIO_U,
                  'n_muestra_por_anio': N_MUESTRA, 'semilla_muestra': SEM_MUESTRA},
    'regla_de_exclusion': 'por origen, no por desempeno',
    'conjuntos': {k: {'n': len(v), 'firma': firma(v), 'variables': sorted(v)}
                  for k, v in VARS.items()},
}

# huellas de los archivos de datos, tomadas del manifiesto del cuaderno 9
try:
    man = pd.read_csv(f'{OUT9}manifiesto_archivos.csv')
    ch = next(c for c in man.columns if any(k in c.lower() for k in ('huella','hash','sha')))
    ca = next(c for c in man.columns if any(k in c.lower() for k in ('archivo','ruta','path','nombre')))
    clave = man[man[ca].astype(str).str.contains('c8_panel|base_maestra_v4|c8_vars|c8_etiqueta',
                                                 na=False, regex=True)]
    esp['archivos'] = clave[[ca, ch]].astype(str).to_dict('records')
    print(f'archivos clave fichados desde el manifiesto: {len(clave)}')
    if len(clave): print(clave[[ca, ch]].to_string(index=False))
except Exception as e:
    esp['archivos'] = f'no disponible: {type(e).__name__}: {e}'
    print('no pude leer el manifiesto:', type(e).__name__, e)

with open(f'{OUT}especificacion_congelada.json', 'w', encoding='utf-8') as f:
    json.dump(esp, f, ensure_ascii=False, indent=2)

print(f'\nEspecificacion congelada el {sello}\n')
print(pd.DataFrame([{'conjunto': k, 'variables': v['n'], 'firma': v['firma']}
                    for k, v in esp['conjuntos'].items()]).to_string(index=False))
print(f'\n  -> {OUT}especificacion_congelada.json')
print('''
Como se usa. Volver a correr esta celda despues de cualquier cambio en las variables.
Si una firma cambia, el conjunto cambio, y ese cambio tiene que quedar escrito en el
registro del cuaderno 10 junto con su efecto sobre el resultado. Una firma igual es
prueba verificable de que el conjunto no se toco entre dos corridas.

Lo que esta celda NO hace: no convierte en preinscripcion algo que no lo fue. Las dos
revisiones de variables que ya ocurrieron estan declaradas arriba con su fecha y su
efecto. Lo que la celda garantiza es que de aqui en adelante cualquier revision deja
rastro.''')

## Pieza 2 · Un índice construido solo con literatura

La objeción concreta sería: `idx_intermediacion` da 0,868, mejor que el modelo entrenado; ¿cómo
sabemos que no se construyó probando combinaciones hasta dar con una que separara a las empresas
fantasma?

La respuesta no puede ser «confíen en nosotros». Hay que construir **otro** índice con una regla
escrita antes de calcular nada, y ver qué da. La regla:

1. **Solo la familia `red_`**, que es «a quién le compra cada entidad». Es también la familia que
   se queda en el tema después de la delimitación con el otro grupo, de modo que el índice
   sobrevive a esa concesión.
2. **Solo componentes con signo teórico inequívoco**: alcance entre compradores
   (`red_n_entidades`), centralidad (`red_autovector`), arraigo en el núcleo (`red_coreness`) y
   volumen (`red_n_contratos`). La hipótesis es una sola y se enuncia completa: *una empresa
   usada como intermediaria aparece conectada a muchos compradores y arraigada en el núcleo de la
   red, en relación con su tamaño.*
3. **Las variables de concentración quedan fuera.** `red_hhi_entidades`,
   `red_cuota_top_entidad`, `red_captura_max` y `red_captura_media` tienen signo teórico ambiguo:
   un intermediario podría estar repartido entre muchos compradores o capturado por uno solo. No
   hay forma de elegir el signo sin mirar el resultado, y mirar el resultado es justo lo que
   estamos tratando de descartar. Excluirlas cuesta información y es el precio de la regla.
4. **Pesos iguales** sobre el percentil dentro del año. No se optimiza nada, ni el peso ni el
   umbral ni el número de componentes.

Y después viene la parte que de verdad decide. Si **cualquier** promedio de variables de red
llegara a 0,87, la elección teórica no estaría haciendo ningún trabajo y el resultado del índice
sería un artefacto de promediar. Así que se construyen 200 índices al azar de la misma familia,
con el mismo número de componentes y signos sorteados, y se mira dónde cae el índice teórico
dentro de esa distribución.


In [ ]:
# --- Pieza 2: el nulo que corresponde a como esta construido el indice ---
# Por que esta version reemplaza la anterior. La primera comparaba idx_intermediacion contra
# indices al azar de CUATRO componentes de la familia red_, rankeados dentro del anio. El indice
# real, construido en la capa 2 parte B5, tiene OCHO componentes de TRES familias y se arma sobre
# el percentil dentro de anio x decil de tamano. Eran dos cosas distintas: el nulo equivocado
# hacia parecer que el indice estaba en el 1 % superior de su propio espacio, cuando ni siquiera
# era su espacio.
#
# Composicion real, copiada de la capa 2. Los signos vienen de la teoria citada alli:
# Burt (agujeros estructurales) y Guimera-Amaral (rol conector) para intermediacion;
# Wachs-Kertesz para cartel. El signo negativo sobre las variables de cierre y concentracion
# es un compromiso teorico, no un parametro libre: un intermediario PUENTEA agujeros, asi que
# agrupamiento y concentracion bajos son parte de la hipotesis.
COMP_INTER = {'red_autovector': +1, 'red_n_entidades': +1, 'cob_n_rivales': +1,
              'com_n_comunidades_rivales': +1, 'com_participacion': +1,
              'cob_agrupamiento': -1, 'red_hhi_entidades': -1, 'cob_hhi_rivales': -1}
COMP_CART  = {'cob_agrupamiento': +1, 'cob_hhi_rivales': +1, 'cob_cuota_rival_top': +1,
              'com_z_intra': +1, 'cob_n_rivales': -1, 'com_participacion': -1}

# El estrato con el que se construyo el indice: anio x decil de tamano, tamano = red_n_contratos
tam = P['red_n_contratos'].fillna(0) if 'red_n_contratos' in P.columns else pd.Series(0, index=P.index)
dec = tam.groupby(P.anio_proceso).transform(
          lambda s: pd.qcut(s.rank(method='first'), 10, labels=False, duplicates='drop'))
P['estrato'] = P.anio_proceso.astype(str) + '_' + dec.astype(str)
y = P.y_fantasma.values.astype(int)
print(f'estratos: {P.estrato.nunique()} (anio x decil de tamano)')

def pct_estrato(x):
    s = P[x] if isinstance(x, str) else pd.Series(np.asarray(x), index=P.index)
    return s.groupby(P.estrato).rank(pct=True).values

def componer(comp):
    """Igual que la capa 2: percentil dentro del estrato; el signo negativo invierte (1 - p)."""
    partes = [(pct_estrato(c) if sg > 0 else 1.0 - pct_estrato(c))
              for c, sg in comp.items() if c in P.columns]
    if not partes: return None, 0
    return promedio_seguro(np.column_stack(partes)), len(partes)

print('componentes de intermediacion ausentes:', [c for c in COMP_INTER if c not in P.columns] or 'ninguno')
print('componentes de cartel ausentes        :', [c for c in COMP_CART  if c not in P.columns] or 'ninguno')

rec_i, ni = componer(COMP_INTER)
rec_c, nc = componer(COMP_CART)
print(f'reconstruidos con {ni} y {nc} componentes')

# Control: la reconstruccion tiene que coincidir con la columna guardada. Si no, el nulo que
# viene abajo no es el nulo de ESTE indice y no hay que leerlo.
for nom, col, rec in [('intermediacion', 'idx_intermediacion', rec_i),
                      ('cartel', 'idx_cartel', rec_c)]:
    if col in P.columns and rec is not None:
        m = P[col].notna().values & ~np.isnan(rec)
        if m.sum() > 10:
            cor = float(np.corrcoef(P[col].values[m], rec[m])[0, 1])
            print(f'  correlacion reconstruccion vs columna guardada ({nom}): {cor:.4f}'
                  + ('   OK' if cor > 0.98 else '   ATENCION: no reproduce la construccion'))

# --- El mismo indice, leido de dos maneras ---
# El paso 12 lo rankea dentro del ANIO. La capa 2 lo construyo y lo leyo dentro de ANIO x DECIL.
# Si el numero baja al estratificar por tamano, parte de lo que mide es tamano, no estructura.
filas = []
for col in ['idx_intermediacion', 'idx_cartel']:
    if col in P.columns:
        filas.append((col, round(stat_indice(pct_anio(P, col), y), 3),
                           round(stat_indice(pct_estrato(col), y), 3)))
if filas:
    print('\n' + pd.DataFrame(filas, columns=['indice', 'pct dentro del anio',
                                              'pct dentro de anio x decil']).to_string(index=False))

# --- El nulo correcto: mismas familias, mismo numero de componentes, mismo estrato ---
N_AZAR = 200
FAM = ('red_', 'cob_', 'com_')
pool = [c for c in P.columns if c.startswith(FAM) and not c.endswith('_pct')]
print(f'\npool del nulo: {len(pool)} variables de las familias {FAM}')
Z = {c: pct_estrato(c) for c in pool}
r = np.random.default_rng(SEMILLA)
anios = P.anio_proceso.values
va, ve = [], []
t0 = time.time()
for b in range(N_AZAR):
    sel = r.choice(len(pool), size=min(max(ni, 2), len(pool)), replace=False)
    sg  = r.choice([-1.0, 1.0], size=len(sel))
    x = promedio_seguro(np.column_stack(
            [Z[pool[j]] if s > 0 else 1.0 - Z[pool[j]] for j, s in zip(sel, sg)]))
    va.append(stat_indice(pct_anio_arr(x, anios), y))
    ve.append(stat_indice(pct_estrato(x), y))
print(f'{N_AZAR} indices al azar de {min(max(ni,2), len(pool))} componentes en {time.time()-t0:.0f} s')

def reporte(nulo, lectura, pcol):
    v = np.array([q for q in nulo if np.isfinite(q)])
    print(f'\n=== Lectura: {lectura} ===')
    print(f'nulo de {len(v)} indices al azar: mediana {np.median(v):.3f} | '
          f'IC 95 % [{np.percentile(v,2.5):.3f}, {np.percentile(v,97.5):.3f}] | '
          f'maximo {v.max():.3f}')
    out = []
    for col in ['idx_intermediacion', 'idx_cartel']:
        if col not in P.columns: continue
        o  = stat_indice(pcol(col), y)
        p1 = (1 + int((v >= o).sum())) / (len(v) + 1)
        p2 = (1 + int((np.abs(v - 0.5) >= abs(o - 0.5)).sum())) / (len(v) + 1)
        out.append((col, round(o, 3), round(p1, 3), round(p2, 3)))
    if out:
        print(pd.DataFrame(out, columns=['indice','observado','p cola superior',
                                         'p dos colas desde 0,5']).to_string(index=False))

reporte(va, 'percentil dentro del anio (como lo evalua el paso 12)', lambda c: pct_anio(P, c))
reporte(ve, 'percentil dentro de anio x decil de tamano (como se construyo)', pct_estrato)

# --- Segundo nulo, mas exigente: los MISMOS ocho componentes, con los signos sorteados ---
# El nulo de arriba pregunta si la construccion completa es especial. Este pregunta algo mas
# estrecho y mas duro: dado que ya elegimos esas ocho variables, cuanto aporta el patron de
# signos que dicta la teoria. Si casi cualquier patron de signos llega a lo mismo, entonces la
# teoria solo eligio las variables; si solo el patron teorico llega, los signos cargan la teoria.
comp_ok = [c for c in COMP_INTER if c in P.columns]
Zi = {c: pct_estrato(c) for c in comp_ok}
signos_reales = tuple(1 if COMP_INTER[c] > 0 else -1 for c in comp_ok)
r2 = np.random.default_rng(SEMILLA + 1)
vs, vistos = [], set()
for b in range(200):
    sg = tuple(int(q) for q in r2.choice([-1, 1], size=len(comp_ok)))
    if sg == signos_reales or sg in vistos: continue
    vistos.add(sg)
    x = promedio_seguro(np.column_stack(
            [Zi[c] if q > 0 else 1.0 - Zi[c] for c, q in zip(comp_ok, sg)]))
    vs.append(stat_indice(pct_estrato(x), y))
vs = np.array([q for q in vs if np.isfinite(q)])
# el valor de comparacion es la RECONSTRUCCION con los signos verdaderos: es el unico
# internamente consistente con los 101 patrones sorteados, que tambien son reconstrucciones
real = stat_indice(pct_estrato(rec_i), y) if rec_i is not None else np.nan
guard = (stat_indice(pct_estrato('idx_intermediacion'), y)
         if 'idx_intermediacion' in P.columns else np.nan)
print(f'\n=== Segundo nulo: los mismos {len(comp_ok)} componentes, {len(vs)} patrones de signos ===')
print(f'nulo: mediana {np.median(vs):.3f} | IC 95 % [{np.percentile(vs,2.5):.3f}, '
      f'{np.percentile(vs,97.5):.3f}] | maximo {vs.max():.3f}')
if np.isfinite(real):
    print(f'patron teorico, reconstruido : {real:.3f}  ->  p cola superior = '
          f'{(1 + int((vs >= real).sum())) / (len(vs) + 1):.3f}')
if np.isfinite(guard):
    print(f'columna guardada en el panel : {guard:.3f}  (debe quedar junto al anterior)')
print(f'cuantos patrones de signos pasan de 0,80: {int((vs > 0.80).sum())} de {len(vs)}')

pd.DataFrame({'azar_anio': va, 'azar_estrato': ve}).to_csv(f'{OUT}indices_azar.csv', index=False)
print(f'\n  -> {OUT}indices_azar.csv')
print('''
Como se lee. Este nulo si es el espacio de construcciones alternativas del indice: mismas
tres familias, mismo numero de componentes, signos sorteados, mismo estrato. Si el indice
queda en la cola de ESTE nulo, la eleccion teorica de componentes y signos esta haciendo
trabajo que un sorteo no hace.

Y el contraste que mas pesa no es el valor p, es idx_cartel. Los dos indices se definieron
juntos, con signos casi espejados sobre cuatro componentes compartidos, y se probo cada uno
UNA vez con correccion por multiplicidad. Si alguien hubiera estado buscando una cifra alta,
no habria dejado publicado un indice hermano apuntando al lado contrario.''')

## Pieza 3 · Especificación por mitades

La tercera forma de circularidad es la más difícil de ver: aunque la regla de exclusión sea
honesta y el índice esté especificado de antemano, la elección entre las cuatro especificaciones
de A —completa, anticipada, estricta, sin ausencia informativa— se hizo viendo los resultados de
las cuatro sobre los mismos 15 conglomerados positivos. Eso es selección, aunque sea suave.

La prueba: partir los conglomerados positivos en dos mitades, elegir la especificación mirando
**solo la primera**, y medir su valor en la segunda, que no se tocó para elegir.

**La primera versión de esta celda estaba mal diseñada y conviene dejarlo escrito.** Partía por
el resto de la huella SHA-256 entre dos, lo cual es reproducible pero no equilibrado: con 15
conglomerados dio 11 contra 4. Con cuatro conglomerados no se estima nada, y la salida lo
gritaba: los cuatro conjuntos de variables, que se diferencian en hasta 95 variables, dieron
entre 0,914 y 0,929. Un rango de 0,015 entre especificaciones tan distintas significa que la
cifra la producían esos cuatro casos y no las variables.

Esta versión reparte alternando sobre la huella, de modo que las mitades son siempre iguales, y
repite la partición doce veces cambiando una sal. Así el resultado no depende de haber tenido
suerte con una partición, y el **rango entre particiones es parte del resultado**, no un detalle.

Con la mitad de los conglomerados, dejar uno fuera sería muy lento, así que esta pieza usa 4
pliegues. Sus cifras **no son comparables con las de la tesis**, que usan dejar-uno-fuera. Lo que
se compara aquí es una mitad contra la otra, con el mismo protocolo en los dos lados.

In [ ]:
# --- Pieza 3: especificacion por mitades, con particion equilibrada y repetida ---
# Por que esta version reemplaza la anterior. La primera partia por el resto de la huella
# entre 2, y con 15 conglomerados eso dio 11 contra 4. Cuatro conglomerados no alcanzan
# para estimar nada, y se vio: los cuatro conjuntos de variables dieron entre 0,914 y 0,929,
# un rango de 0,015 entre especificaciones que se diferencian en 95 variables. Esa cifra la
# producian esos cuatro casos, no las variables.
#
# Aqui la particion es SIEMPRE por mitades iguales y se repite R_PART veces cambiando una sal,
# de modo que el resultado no dependa de una particion afortunada.
R_PART, N_PLIEGUES = 12, 4

def mitades_equilibradas(cl, sal):
    """Ordena los conglomerados por la huella de (sal, ruc) y los reparte alternando.
    Siempre quedan mitades iguales, y con la misma sal cualquiera reproduce la particion."""
    h = sorted(cl, key=lambda x: hashlib.sha256(f'{sal}:{x}'.encode('utf-8')).hexdigest())
    return h[0::2], h[1::2]

pos_cl = sorted(P.loc[P.y_fantasma == 1, 'proveedor_ruc'].astype(str).unique())
rucs   = P.proveedor_ruc.astype(str)
conjuntos = [k for k in ['A', 'A_anticipada', 'A_estricta', 'A_sin_ausencia'] if k in VARS]
COLS = {k: [c for c in VARS[k] if c in P.columns] for k in conjuntos}
print(f'conglomerados positivos: {len(pos_cl)} | {R_PART} particiones de '
      f'{len(pos_cl) - len(pos_cl)//2} y {len(pos_cl)//2} conglomerados')
print(f'conjuntos comparados: {conjuntos}\n')

def medir(cols, fuera):
    """Estadistico sobre el panel sin los positivos de la otra mitad."""
    d  = P[~rucs.isin(fuera)]
    yy = d.y_fantasma.values.astype(int)
    if yy.sum() == 0 or len(np.unique(d.proveedor_ruc.values[yy == 1])) < 2:
        return np.nan
    X = d[cols].astype(float).values
    return estadistico(X, yy, pd.factorize(d.proveedor_ruc.values)[0],
                       d.anio_proceso.values, N_PLIEGUES, K_P, SEMILLA, Xu=X[yy == 0])

filas = []
t0 = time.time()
for s in range(R_PART):
    m1, m2 = mitades_equilibradas(pos_cl, s)
    v1  = {k: medir(COLS[k], m2) for k in conjuntos}          # mitad 1: aqui se ELIGE
    gan = max(v1, key=lambda k: v1[k] if np.isfinite(v1[k]) else -1.0)
    v2g = medir(COLS[gan], m1)                                # mitad 2: solo se MIDE
    v2a = v2g if gan == 'A' else medir(COLS['A'], m1)
    filas.append((s, len(m1), len(m2), gan, round(v1[gan], 3), round(v2g, 3), round(v2a, 3)))
    print(f'  particion {s+1:2d}/{R_PART}: gana {gan:15s} m1 {v1[gan]:.3f} -> m2 {v2g:.3f} '
          f'| A en m2 {v2a:.3f}   ({time.time()-t0:.0f} s)')

t = pd.DataFrame(filas, columns=['particion','n_m1','n_m2','ganador_en_m1',
                                 'valor_m1','ganador_en_m2','A_en_m2'])
print('\nCon que frecuencia gana cada conjunto al mirar solo la mitad 1:')
print(t.ganador_en_m1.value_counts().to_string())
print()
for col, et in [('valor_m1','ganador, en la mitad donde se eligio'),
                ('ganador_en_m2','ganador, en la mitad que no se uso'),
                ('A_en_m2','conjunto A, en la mitad que no se uso')]:
    x = t[col].astype(float).dropna()
    if len(x):
        print(f'  {et:40s} mediana {x.median():.3f} | [{x.min():.3f} - {x.max():.3f}] '
              f'| rango {x.max()-x.min():.3f}')

t.to_csv(f'{OUT}mitades.csv', index=False)
print(f'\n  -> {OUT}mitades.csv')
print('''
Como se lee, en dos preguntas separadas.

1) Si el conjunto ganador CAMBIA de una particion a otra, entonces la eleccion entre las
   cuatro especificaciones es ruido, y la tesis no puede afirmar que una sea mejor: tiene
   que reportar las cuatro. Eso no es un fracaso, es un limite del tamano de la muestra.

2) Si el valor en la mitad que no se uso para elegir es sistematicamente mas bajo que en la
   mitad donde se eligio, hay optimismo por seleccion y la cifra honesta es la segunda.

El rango entre particiones es la incertidumbre real de esta prueba. Con 15 conglomerados
partidos en dos, va a ser grande, y declararlo es parte del resultado.''')

## Cierre · Qué queda respondido y qué no

**Respondido.** Las tres piezas, juntas, dicen esto: los conjuntos de variables están firmados y
cualquier cambio futuro queda a la vista; existe un índice con regla de construcción escrita que
se evalúa contra un nulo de índices al azar de la misma familia; y la elección de especificación
se puede verificar en una mitad de los positivos que no se usó para elegirla.

**No respondido, y hay que decirlo en la tesis.** La especificación no fue preinscrita. Las dos
revisiones de variables ocurrieron después de ver resultados y están declaradas en la pieza 1 con
su fecha y su dirección de efecto. El argumento que sostiene el trabajo no es «no mirábamos», es
**«miramos, lo escribimos, y la dirección del efecto es la contraria a la que produce elegir para
que salga»**. Un tribunal puede aceptar eso o no, pero es lo que ocurrió, y afirmar otra cosa
sería una falla peor que las quince del cuaderno 10.

**Lo que haría falta para cerrarlo de verdad**, si hubiera tiempo: fijar la especificación hoy,
con la firma de la pieza 1, y validarla contra un conjunto de positivos que todavía no existe
—empresas que el SRI marque como fantasma en los próximos meses—. Eso sí sería preinscripción, y
es la línea natural de trabajo futuro para el capítulo de conclusiones.


In [ ]:
# --- diagnostico: que componente de intermediacion no reproduce ---
g = P.idx_intermediacion.values
comp_ok = [c for c in COMP_INTER if c in P.columns]

def corr_con(sub):
    x = promedio_seguro(np.column_stack(
        [(pct_estrato(c) if COMP_INTER[c] > 0 else 1.0 - pct_estrato(c)) for c in sub]))
    m = ~pd.isna(g) & ~np.isnan(x)
    return float(np.corrcoef(g[m].astype(float), x[m])[0, 1]), int(m.sum())

base, n = corr_con(comp_ok)
print(f'con los {len(comp_ok)} componentes: r = {base:.4f}   ({n:,} filas comparadas)')
print('\nquitando uno a la vez. Si r SUBE, ese es el que no cuadra:')
fil = []
for c in comp_ok:
    rr, _ = corr_con([x for x in comp_ok if x != c])
    fil.append((c, COMP_INTER[c], round(rr, 4), round(rr - base, 4),
                round(float(pd.isna(P[c]).mean() * 100), 1)))
print(pd.DataFrame(fil, columns=['componente_quitado','signo','r_sin_el','cambio','pct_nulo'])
        .sort_values('cambio', ascending=False).to_string(index=False))

# la capa 2 guardo cuantos componentes tenia cada fila: si esa columna esta, es la prueba directa
if 'n_comp_intermediacion' in P.columns:
    mio = np.sum(np.column_stack([~pd.isna(P[c]).values for c in comp_ok]), axis=1)
    cmp_ = pd.crosstab(P.n_comp_intermediacion, mio)
    print('\ncomponentes por fila: capa 2 (filas) vs reconstruccion (columnas)')
    print(cmp_.to_string())
    print('Si no es diagonal, la diferencia esta en el patron de nulos, no en los valores.')
else:
    print('\nn_comp_intermediacion no esta en el panel; se puede leer de indices_red.parquet')